In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
%pip install xgboost
%pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.8 MB/s eta 0:00:00


In [ ]:
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import cross_val_score, StratifiedKFold
import optuna

In [ ]:
train=pd.read_csv('train.csv')
test=pd.read_csv('test.csv')

In [ ]:
train=train.drop('id', axis=1).copy()
id_test=test['id']
X_test=test.drop('id', axis=1).copy()

y_train=train['Irrigation_Need']
X_train=train.drop('Irrigation_Need', axis=1)

cat_features = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
    'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']

num_cols = [col for col in X_train.columns if col not in cat_features]

#Impute any missing values
my_imputer=SimpleImputer(strategy='median')
X_train[num_cols]=my_imputer.fit_transform(X_train[num_cols])
X_test[num_cols]=my_imputer.transform(X_test[num_cols])

cat_imputer = SimpleImputer(strategy='constant', fill_value='Missing')
X_train[cat_features] = cat_imputer.fit_transform(X_train[cat_features])
X_test[cat_features] = cat_imputer.transform(X_test[cat_features])

#Add new insights and avoid division by zero
X_train['Climate_Stress']=X_train['Temperature_C']/(X_train['Humidity']+1)
X_test['Climate_Stress']=X_test['Temperature_C']/(X_test['Humidity']+1)
X_train['Water_Source_Efficiency'] = X_train['Soil_Moisture'] + X_train['Rainfall_mm']
X_test['Water_Source_Efficiency'] = X_test['Soil_Moisture'] + X_test['Rainfall_mm']

for col in cat_features:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

#verification
print("Checking Dtypes again (Must be 'category'):")
print(X_train[cat_features].dtypes)

#Ordinal encoding
y_train=y_train.map({'Low':0, 'Medium':1, 'High':2})

Checking Dtypes again (Must be 'category'):
Soil_Type            category
Crop_Type            category
Crop_Growth_Stage    category
Season               category
Irrigation_Type      category
Water_Source         category
Mulching_Used        category
Region               category
dtype: object


In [ ]:
def objective(trial):
    # Parameters to tune
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'device': 'cuda',
        'tree_method': 'hist',
        'enable_categorical': True # Since we set columns to 'category' earlier
    }

    model = XGBClassifier(**param)

    # Use Stratified K-Fold for a robust score
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_train, y_train, cv=cv, scoring='balanced_accuracy')

    return score.mean()

In [ ]:
# Create the study and optimize
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50) # Start with 50 trials

[I 2026-04-24 10:44:30,562] A new study created in memory with name: no-name-3aca822f-7673-4c51-9b28-c8b8d829f62e
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [10:45:13] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
[I 2026-04-24 10:47:51,090] Trial 0 finished with value: 0.9611673786078704 and parameters: {'n_estimators': 653, 'max_depth': 10, 'learning_rate': 0.03108607446212198}. Best is trial 0 with value: 0.9611673786078704.
[I 2026-04-24 10:52:28,421] Trial 1 finished with value: 0.9613365748809357 and parameters: {'n_estimators': 1010

In [ ]:
print("Best Score:", study.best_value)
print("Best Params:", study.best_params)

final_model = XGBClassifier(**study.best_params,
                            tree_method='hist',
                            enable_categorical=True,
                            device='cuda'
                            )
final_model.fit(X_train, y_train)

Best Score: 0.962637465255011
Best Params: {'n_estimators': 1696, 'max_depth': 3, 'learning_rate': 0.09830674121193768}


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device='cuda', early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.09830674121193768, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1696, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
#Make predictions
predictions = final_model.predict(X_test)
df=pd.DataFrame()
df['id']=id_test
df['Irrigation_Need']=predictions
df['Irrigation_Need']=df['Irrigation_Need'].map({0:'Low', 1:'Medium', 2:'High'})
df.to_csv('my_prediction6.csv',index=False)